In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
columns = [
    'checking_account',     # Status of existing checking account
    'duration',             # Duration in months
    'credit_history',       # Credit history
    'purpose',              # Purpose of loan
    'credit_amount',        # Credit amount
    'savings_account',      # Savings account/bonds
    'employment_since',     # Present employment since
    'installment_rate',     # Installment rate (% of disposable income)
    'personal_status',      # Personal status and sex
    'other_debtors',        # Other debtors/guarantors
    'residence_since',      # Present residence since
    'property',             # Property
    'age',                  # Age in years
    'other_installments',   # Other installment plans
    'housing',              # Housing
    'existing_credits',     # Number of existing credits at this bank
    'job',                  # Job
    'liable_people',        # Number of people liable to provide maintenance
    'telephone',            # Telephone
    'foreign_worker',       # Foreign worker
    'target'                # 1 = Good, 2 = Bad
]

In [3]:
df = pd.read_csv('../data/original/german_credit_data.csv')
print(f"Dataset loaded: {df.shape[0]} rows, {df.shape[1]} columns ✓")

Dataset loaded: 1000 rows, 21 columns ✓


In [4]:
# Shape
print("=== SHAPE ===")
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")

# Data types
print("\n=== DATA TYPES ===")
print(df.dtypes)

# First look
print("\n=== FIRST 5 ROWS ===")
df.head()

=== SHAPE ===
Rows: 1000, Columns: 21

=== DATA TYPES ===
checking_status             str
duration                  int64
credit_history              str
purpose                     str
credit_amount             int64
savings_status              str
employment                  str
installment_commitment    int64
personal_status             str
other_parties               str
residence_since           int64
property_magnitude          str
age                       int64
other_payment_plans         str
housing                     str
existing_credits          int64
job                         str
num_dependents            int64
own_telephone               str
foreign_worker              str
class                       str
dtype: object

=== FIRST 5 ROWS ===


,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


check for comma-seperated or space-seperated csv file


In [5]:
# First, peek at the raw file to confirm the separator
with open('../data/original/german_credit_data.csv', 'r') as f:
    for i, line in enumerate(f):
        print(repr(line))
        if i == 3:
            break

'checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,residence_since,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class\n'
'<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,4,real estate,67,none,own,2,skilled,1,yes,yes,good\n'
'0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,2,real estate,22,none,own,1,skilled,1,none,yes,bad\n'
'no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,3,real estate,49,none,own,1,unskilled resident,2,none,yes,good\n'


In [6]:
print("=== MISSING VALUES ===")
print(df.isnull().sum())

print(f"\n=== DUPLICATES ===")
print(f"Duplicate rows: {df.duplicated().sum()}")

=== MISSING VALUES ===
checking_status           0
duration                  0
credit_history            0
purpose                   0
credit_amount             0
savings_status            0
employment                0
installment_commitment    0
personal_status           0
other_parties             0
residence_since           0
property_magnitude        0
age                       0
other_payment_plans       0
housing                   0
existing_credits          0
job                       0
num_dependents            0
own_telephone             0
foreign_worker            0
class                     0
dtype: int64

=== DUPLICATES ===
Duplicate rows: 0


In [7]:
print("=== TARGET DISTRIBUTION ===")
print(df['class'].value_counts())
print(f"\nClass balance: {df['class'].value_counts(normalize=True).round(3) * 100}")

=== TARGET DISTRIBUTION ===
class
good    700
bad     300
Name: count, dtype: int64

Class balance: class
good    70.0
bad     30.0
Name: proportion, dtype: float64


In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
numeric_cols 

['duration',
 'credit_amount',
 'installment_commitment',
 'residence_since',
 'age',
 'existing_credits',
 'num_dependents']

Truly continuous => duration, credit_amount, age

Ordinal numeric => installment_commitment, residence_since, existing_credits, num_dependants



In [9]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()
categorical_cols = categorical_cols[:-1]  # Exclude target
categorical_cols

['checking_status',
 'credit_history',
 'purpose',
 'savings_status',
 'employment',
 'personal_status',
 'other_parties',
 'property_magnitude',
 'other_payment_plans',
 'housing',
 'job',
 'own_telephone',
 'foreign_worker']

## Univariate EDA

Univariate means one variable at a time. like  asking: "What does this variable look like on its own?"
For continuous variables we look at:

- Distribution — is it normal, skewed, bimodal?
- Outliers — are there extreme values?
- Central tendency — where does most data cluster?

In [10]:
df[numeric_cols].describe().round(2)

,duration,credit_amount,installment_commitment,residence_since,age,existing_credits,num_dependents
count,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00,1000.00
mean,20.90,3271.26,2.97,2.84,35.55,1.41,1.16
std,12.06,2822.74,1.12,1.10,11.38,0.58,0.36
min,4.00,250.00,1.00,1.00,19.00,1.00,1.00
25%,12.00,1365.50,2.00,2.00,27.00,1.00,1.00
50%,18.00,2319.50,3.00,3.00,33.00,1.00,1.00
75%,24.00,3972.25,4.00,4.00,42.00,2.00,1.00
max,72.00,18424.00,4.00,4.00,75.00,4.00,2.00


In [ ]:
continuous_cols = ['duration', 'credit_amount', 'age']

for col in continuous_cols:
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f'{col} — Distribution', 
                                        f'{col} — Boxplot'))
    
    # Histogram
    fig.add_trace(
        go.Histogram(x=df[col], nbinsx=30,
                     marker_color='steelblue', opacity=0.75,
                     name='Distribution'),
        row=1, col=1
    )
    
    # Boxplot
    fig.add_trace(
        go.Box(y=df[col], marker_color='steelblue',
               name='Boxplot', boxmean=True),
        row=1, col=2
    )
    
    fig.update_layout(
        title=f'Univariate Analysis — {col}',
        height=400, showlegend=False,
        template='plotly_dark'
    )
    fig.show()

In [12]:
# Check the actual outlier count for credit_amount
Q1 = df['credit_amount'].quantile(0.25)
Q3 = df['credit_amount'].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df['credit_amount'] < Q1 - 1.5*IQR) | 
              (df['credit_amount'] > Q3 + 1.5*IQR)]

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Outlier count: {len(outliers)}")
print(f"Outlier % : {len(outliers)/len(df)*100:.1f}%")

Q1: 1365.5, Q3: 3972.25, IQR: 2606.75
Outlier count: 72
Outlier % : 7.2%


In [13]:
ordinal_cols = ['installment_commitment', 'residence_since', 
                'existing_credits', 'num_dependents']

for col in ordinal_cols:
    counts = df[col].value_counts().sort_index()
    
    fig = px.bar(x=counts.index, y=counts.values,
                 labels={'x': col, 'y': 'Count'},
                 title=f'Univariate Analysis — {col}',
                 color=counts.values,
                 color_continuous_scale='Blues',
                 template='plotly_dark')
    
    fig.update_layout(height=350, showlegend=False)
    fig.show()

## Univariate EDA — Key Findings

| Variable | Finding | Action |
|---|---|---|
| credit_amount | Right skewed, 72 outliers (7.2%) | Log transform on Day 2 |
| duration | Skewed, clusters at 12–24 months | Check long loans vs default rate |
| age | Slight right skew, outliers at high end | Monitor in bivariate |
| installment_rate | Most borrowers at rate=4 (highest burden) | Strong risk signal — test in hypothesis testing |
| existing_credits | 333 people have 2 existing loans | Engineer debt burden feature on Day 2 |

In [14]:
import os
os.makedirs('../reports/figures', exist_ok=True)

def save_plot(fig, filename):
    """Call this on Day 7 to export any plot"""
    fig.write_image(f'../reports/figures/{filename}.png')

In [15]:
#  how many unique categories each column has
for col in categorical_cols:
    print(f"{col:25s} → {df[col].nunique()} unique values: {df[col].unique()}")





checking_status           → 4 unique values: <StringArray>
['<0', '0<=X<200', 'no checking', '>=200']
Length: 4, dtype: str
credit_history            → 5 unique values: <StringArray>
['critical/other existing credit',                  'existing paid',
             'delayed previously',            'no credits/all paid',
                       'all paid']
Length: 5, dtype: str
purpose                   → 10 unique values: <StringArray>
[           'radio/tv',           'education', 'furniture/equipment',
             'new car',            'used car',            'business',
  'domestic appliance',             'repairs',               'other',
          'retraining']
Length: 10, dtype: str
savings_status            → 5 unique values: <StringArray>
['no known savings', '<100', '500<=X<1000', '>=1000', '100<=X<500']
Length: 5, dtype: str
employment                → 5 unique values: <StringArray>
['>=7', '1<=X<4', '4<=X<7', 'unemployed', '<1']
Length: 5, dtype: str
personal_status           →

# rank of the importance of below attributes based on above analysis (with general idea)
1. checking_status => Low or no balance strongly correlates with inability to repay short-term credits.
2. credit_history => past payment behaviour builds opinion on future payment behaviour 
3. housing => mirror of financial stability and asset (own>rent>free)
4. purpose => risk is context-dependant here

In [16]:
priority_cats = ['checking_status', 'credit_history', 'housing', 'purpose']

for col in priority_cats:
    counts = df[col].value_counts()
    
    fig = px.bar(x=counts.index, y=counts.values,
                 labels={'x': col, 'y': 'Count'},
                 title=f'Univariate Analysis — {col}',
                 color=counts.values,
                 color_continuous_scale='Blues',
                 template='plotly_dark',
                 text=counts.values)
    
    fig.update_traces(textposition='outside')
    fig.update_layout(height=500, showlegend=False)
    fig.show()

In [17]:
remaining_cats = ['savings_status', 'employment', 'personal_status',
                  'other_parties', 'property_magnitude', 
                  'other_payment_plans', 'job',
                  'own_telephone', 'foreign_worker']

fig = make_subplots(rows=3, cols=3,
                    subplot_titles=remaining_cats)

for idx, col in enumerate(remaining_cats):
    row = idx // 3 + 1
    col_pos = idx % 3 + 1
    counts = df[col].value_counts()
    
    fig.add_trace(
        go.Bar(x=counts.index, y=counts.values,
               marker_color='steelblue',
               text=counts.values,
               textposition='outside',
               name=col),
        row=row, col=col_pos
    )

fig.update_layout(height=1000,showlegend=False,
                  title='Remaining Categorical Variables',
                  template='plotly_dark')
fig.show()

## Categorical Univariate — Key Findings

| Variable | Finding | Watch For |
|---|---|---|
| checking_status | 66.8% absent or negative balance | Strongest default signal — test with Chi-square |
| credit_history | existing paid most common, but critical/other exists | Bivariate — does critical/other default more? |
| housing | 71.3% own — majority have assets | free housing group default rate |
| purpose | radio/tv dominates, repairs/retraining are distress signals | Chi-square test — purpose vs default |
| savings_status | <100 DM most common — low financial buffer | Combine with checking_account for stress score on Day 2 |
| employment | 1-4 years dominates | Expected given age distribution |
| foreign_worker | 96.3% yes — near zero variance | Candidate for dropping on Day 2, fairness flag |

## Bivariate EDA

In [18]:

# df['class'] = df['class'].map({'good': 1, 'bad': 2})


In [19]:
df.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,critical/other existing credit,radio/tv,1169,no known savings,>=7,4,male single,none,...,real estate,67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,existing paid,radio/tv,5951,<100,1<=X<4,2,female div/dep/mar,none,...,real estate,22,none,own,1,skilled,1,none,yes,bad
2,no checking,12,critical/other existing credit,education,2096,<100,4<=X<7,2,male single,none,...,real estate,49,none,own,1,unskilled resident,2,none,yes,good
3,<0,42,existing paid,furniture/equipment,7882,<100,4<=X<7,2,male single,guarantor,...,life insurance,45,none,for free,1,skilled,2,none,yes,good
4,<0,24,delayed previously,new car,4870,<100,1<=X<4,3,male single,none,...,no known property,53,none,for free,2,skilled,2,none,yes,bad


In [20]:
print(df['class'].value_counts())

class
good    700
bad     300
Name: count, dtype: int64


In [21]:
for col in continuous_cols:
    fig = px.histogram(df, x=col, color='class',
                       barmode='overlay',
                       opacity=0.7,
                       color_discrete_map={'good': 'steelblue', 'bad': 'crimson'},
                       title=f'{col} — Distribution by Credit Risk',
                       template='plotly_dark',
                       marginal='box')
    
    fig.update_layout(height=450)
    fig.show()

In [22]:
print("=== MEAN BY class ===")
print(df.groupby('class')[continuous_cols].mean().round(2))

print("\n=== MEDIAN BY class ===")
print(df.groupby('class')[continuous_cols].median().round(2))

=== MEAN BY class ===
       duration  credit_amount    age
class                                
bad       24.86        3938.13  33.96
good      19.21        2985.46  36.22

=== MEDIAN BY class ===
       duration  credit_amount   age
class                               
bad        24.0         2574.5  31.0
good       18.0         2244.0  34.0


In [23]:
priority_cats = ['checking_status', 'credit_history', 'housing', 'purpose']

for col in priority_cats:
    # Calculate default rate per category
    default_rate = df.groupby(col)['class'].apply(
        lambda x: (x == 'bad').sum() / len(x) * 100
    ).reset_index()
    default_rate.columns = [col, 'default_rate']
    default_rate = default_rate.sort_values('default_rate', ascending=False)
    
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=(f'{col} — Count by Risk',
                                        f'{col} — Default Rate %'))
    
    # Left: stacked count
    for label, color in [('good', 'steelblue'), ('bad', 'crimson')]:
        subset = df[df['class'] == label][col].value_counts()
        fig.add_trace(
            go.Bar(name=label, x=subset.index, y=subset.values,
                   marker_color=color, opacity=0.8),
            row=1, col=1
        )
    
    # Right: default rate
    fig.add_trace(
        go.Bar(x=default_rate[col], y=default_rate['default_rate'],
               marker_color='crimson', opacity=0.8,
               text=default_rate['default_rate'].round(1),
               textposition='outside',
               name='Default Rate %'),
        row=1, col=2
    )
    
    fig.update_layout(height=450, barmode='stack',
                      title=f'Bivariate Analysis — {col} vs class',
                      template='plotly_dark')
    fig.show()

In [24]:
print(df.groupby('credit_history')['class'].value_counts())
print("\nDefault rate by credit history:")
print(df.groupby('credit_history')['class'].apply(
    lambda x: (x == 'bad').sum() / len(x) * 100).round(1))

credit_history                  class
all paid                        bad       28
                                good      21
critical/other existing credit  good     243
                                bad       50
delayed previously              good      60
                                bad       28
existing paid                   good     361
                                bad      169
no credits/all paid             bad       25
                                good      15
Name: count, dtype: int64

Default rate by credit history:
credit_history
all paid                          57.1
critical/other existing credit    17.1
delayed previously                31.8
existing paid                     31.9
no credits/all paid               62.5
Name: class, dtype: float64


## Bivariate EDA — Credit History Key Finding

- "critical/other" has LOWEST default rate (17.1%) despite alarming label
  → Experienced multi-credit borrowers manage debt better
- "all paid" has HIGH default rate (57.1%) — no active credit = no recent track record
- "no credits/all paid" — 62.5% but only 40 samples — statistically unreliable
- Action: Hypothesis test (Chi-square) to validate these differences on Day 1

In [25]:
remaining_cats = ['savings_status', 'employment', 'personal_status',
                  'other_parties', 'property_magnitude',
                  'other_payment_plans', 'job',
                  'own_telephone', 'foreign_worker']

fig = make_subplots(rows=3, cols=3,
                    subplot_titles=remaining_cats)

for idx, col in enumerate(remaining_cats):
    row = idx // 3 + 1
    col_pos = idx % 3 + 1
    
    default_rate = df.groupby(col)['class'].apply(
        lambda x: (x == 'bad').sum() / len(x) * 100
    ).reset_index()
    default_rate.columns = [col, 'default_rate']
    default_rate = default_rate.sort_values('default_rate', ascending=False)
    
    fig.add_trace(
        go.Bar(x=default_rate[col],
               y=default_rate['default_rate'],
               marker_color='crimson',
               opacity=0.8,
               text=default_rate['default_rate'].round(1),
               textposition='outside',
               name=col),
        row=row, col=col_pos
    )

fig.update_layout(height=900, showlegend=False,
                  title='Remaining Categoricals — Default Rate %',
                  template='plotly_dark')
fig.show()

## Bivariate EDA — Remaining Categoricals

Strong discriminators (gap > 20%):
- other_parties, savings_status, property_magnitude

Moderate discriminators (10-20%):
- employment, personal_status, other_payment_plans, foreign_worker*

Weak — drop candidates on Day 2:
- job (6.5%), own_telephone (3%)

*foreign_worker: large gap but n=37 native workers — statistically unreliable + fairness flag

## EDA Conclusion — Default Borrower Profile

A borrower most likely to default shows these characteristics:
1. checking_status < 0 or 0<=X<200 — already in financial stress
2. credit_history: all paid / no credits — no active repayment track record
3. other_parties: co-applicant — shared responsibility dilutes accountability
4. savings_status: <100 or <500 DM — no financial buffer
5. property_magnitude: no known property — no collateral, no asset
6. duration >= 45 months — long loan = high uncertainty
7. age ~ 25-30 — early career, less financial stability


## Multivariate EDA

In [26]:
# Encode target as numeric for correlation
df['class_numeric'] = df['class'].map({'good': 0, 'bad': 1})  # 0=Good, 1=Bad

corr_matrix = df[numeric_cols + ['class_numeric']].corr().round(2)

fig = px.imshow(corr_matrix,
                text_auto=True,
                color_continuous_scale='RdBu_r',
                title='Correlation Heatmap — Numeric Features vs Target',
                template='plotly_dark',
                aspect='auto')

fig.update_layout(height=500)
fig.show()

In [27]:
interaction = df.groupby(['checking_status', 'credit_history'])['class'].apply(
    lambda x: (x == 'bad').sum() / len(x) * 100
).reset_index()
interaction.columns = ['checking_status', 'credit_history', 'default_rate']

fig = px.density_heatmap(interaction,
                          x='checking_status',
                          y='credit_history',
                          z='default_rate',
                          color_continuous_scale='Reds',
                          title='Interaction — Checking Account × Credit History vs Default Rate %',
                          template='plotly_dark',
                          text_auto=True)

fig.update_layout(height=500)
fig.show()

## Multivariate EDA — Key Findings

### Correlation:
- credit_amount × duration: 0.62 — multicollinearity
  → Action: engineer credit_amount/duration = monthly burden feature on Day 2
- No severe multicollinearity elsewhere (no pair > 0.7)

### Interaction — checking_account × credit_history:
- <0 checking + no credits = 76.92% default rate
- Risk compounds, not just adds
  → Action: engineer interaction feature on Day 2

In [28]:
# What correlates most with target?
print(corr_matrix['class_numeric'].sort_values(ascending=False))

class_numeric             1.00
duration                  0.21
credit_amount             0.15
installment_commitment    0.07
residence_since           0.00
num_dependents           -0.00
existing_credits         -0.05
age                      -0.09
Name: class_numeric, dtype: float64


## Correlation with Target — Key Finding

- All numeric correlations < 0.25 — not a linearly separable problem
- duration (0.21) strongest numeric predictor
- residence_since, num_dependents ~ 0 — drop candidates on Day 2
- Signal lives in categorical features + feature interactions
- Implication: tree-based models (XGBoost) preferred over linear models




✅ Univariate EDA    — distributions, outliers, skew
✅ Bivariate EDA     — every feature vs target
✅ Multivariate EDA  — correlations + interactions

In [29]:
# These have near-zero correlation AND near-zero bivariate gap
drop_candidates = ['residence_since', 'num_dependents']
print("Drop candidates from numeric features:", drop_candidates)

Drop candidates from numeric features: ['residence_since', 'num_dependents']


## Statistical Testing

In [30]:
from scipy import stats

print("=" * 55)
print("MANN-WHITNEY U TEST — NUMERIC FEATURES VS TARGET")
print("=" * 55)

good = df[df['class'] == 'good']
bad = df[df['class'] == 'bad']

for col in numeric_cols:
    stat, p_value = stats.mannwhitneyu(good[col], bad[col],
                                        alternative='two-sided')
    significance = "✅ SIGNIFICANT" if p_value < 0.05 else "❌ NOT SIGNIFICANT"
    print(f"\n{col}")
    print(f"  H0: No difference in {col} between Good and Bad borrowers")
    print(f"  p-value: {p_value:.4f} → {significance}")

MANN-WHITNEY U TEST — NUMERIC FEATURES VS TARGET

duration
  H0: No difference in duration between Good and Bad borrowers
  p-value: 0.0000 → ✅ SIGNIFICANT

credit_amount
  H0: No difference in credit_amount between Good and Bad borrowers
  p-value: 0.0059 → ✅ SIGNIFICANT

installment_commitment
  H0: No difference in installment_commitment between Good and Bad borrowers
  p-value: 0.0199 → ✅ SIGNIFICANT

residence_since
  H0: No difference in residence_since between Good and Bad borrowers
  p-value: 0.9358 → ❌ NOT SIGNIFICANT

age
  H0: No difference in age between Good and Bad borrowers
  p-value: 0.0004 → ✅ SIGNIFICANT

existing_credits
  H0: No difference in existing_credits between Good and Bad borrowers
  p-value: 0.1348 → ❌ NOT SIGNIFICANT

num_dependents
  H0: No difference in num_dependents between Good and Bad borrowers
  p-value: 0.9242 → ❌ NOT SIGNIFICANT


In [31]:
from scipy.stats import chi2_contingency

print("=" * 55)
print("CHI-SQUARE TEST — CATEGORICAL FEATURES VS TARGET")
print("=" * 55)

for col in categorical_cols:
    contingency_table = pd.crosstab(df[col], df['class'])
    chi2, p_value, dof, expected = chi2_contingency(contingency_table)
    significance = "✅ SIGNIFICANT" if p_value < 0.05 else "❌ NOT SIGNIFICANT"
    print(f"\n{col}")
    print(f"  H0: {col} is independent of credit risk")
    print(f"  chi2: {chi2:.2f}, dof: {dof}, p-value: {p_value:.4f} → {significance}")

CHI-SQUARE TEST — CATEGORICAL FEATURES VS TARGET

checking_status
  H0: checking_status is independent of credit risk
  chi2: 123.72, dof: 3, p-value: 0.0000 → ✅ SIGNIFICANT

credit_history
  H0: credit_history is independent of credit risk
  chi2: 61.69, dof: 4, p-value: 0.0000 → ✅ SIGNIFICANT

purpose
  H0: purpose is independent of credit risk
  chi2: 33.36, dof: 9, p-value: 0.0001 → ✅ SIGNIFICANT

savings_status
  H0: savings_status is independent of credit risk
  chi2: 36.10, dof: 4, p-value: 0.0000 → ✅ SIGNIFICANT

employment
  H0: employment is independent of credit risk
  chi2: 18.37, dof: 4, p-value: 0.0010 → ✅ SIGNIFICANT

personal_status
  H0: personal_status is independent of credit risk
  chi2: 9.61, dof: 3, p-value: 0.0222 → ✅ SIGNIFICANT

other_parties
  H0: other_parties is independent of credit risk
  chi2: 6.65, dof: 2, p-value: 0.0361 → ✅ SIGNIFICANT

property_magnitude
  H0: property_magnitude is independent of credit risk
  chi2: 23.72, dof: 3, p-value: 0.0000 → ✅ 

In [33]:
# Collect all results
results = []

# Numeric
for col in numeric_cols:
    stat, p = stats.mannwhitneyu(good[col], bad[col], alternative='two-sided')
    results.append({'feature': col, 'test': 'Mann-Whitney U', 'p_value': p})

# Categorical
for col in categorical_cols:
    contingency_table = pd.crosstab(df[col], df['class'])
    chi2, p, dof, expected = chi2_contingency(contingency_table)
    results.append({'feature': col, 'test': 'Chi-Square', 'p_value': p})

results_df = pd.DataFrame(results).sort_values('p_value')

# Plot
fig = px.bar(results_df,
             x='feature', y='p_value',
             color='test',
             title='Hypothesis Test Results — p-values by Feature',
             template='plotly_dark',
             text=results_df['p_value'].round(3),
             color_discrete_map={'Mann-Whitney U': 'steelblue',
                                 'Chi-Square': 'crimson'})

fig.add_hline(y=0.05, line_dash='dash', line_color='yellow',
              annotation_text='p=0.05 threshold')

fig.update_layout(height=700)
fig.update_traces(textposition='outside')
fig.show()

print("\nFeatures BELOW p=0.05 (statistically significant):")
print(results_df[results_df['p_value'] < 0.05]['feature'].tolist())

print("\nFeatures ABOVE p=0.05 (not significant — drop candidates):")
print(results_df[results_df['p_value'] >= 0.05]['feature'].tolist())


Features BELOW p=0.05 (statistically significant):
['checking_status', 'credit_history', 'duration', 'savings_status', 'property_magnitude', 'housing', 'purpose', 'age', 'employment', 'other_payment_plans', 'credit_amount', 'foreign_worker', 'installment_commitment', 'personal_status', 'other_parties']

Features ABOVE p=0.05 (not significant — drop candidates):
['existing_credits', 'own_telephone', 'job', 'num_dependents', 'residence_since']


In [34]:
drop_candidates = ['residence_since', 'num_dependents', 
                   'existing_credits', 'own_telephone', 'job']

print("Confirmed drop candidates for Day 2:")
for f in drop_candidates:
    print(f"  - {f}")

Confirmed drop candidates for Day 2:
  - residence_since
  - num_dependents
  - existing_credits
  - own_telephone
  - job


## Hypothesis Testing — Results Summary

### Mann-Whitney U (Numeric Features):
| Feature | Result | Action |
|---|---|---|
| duration | ✅ Significant (p≈0) | Keep — strongest numeric feature |
| credit_amount | ✅ Significant | Keep — combine with duration on Day 2 |
| age | ✅ Significant | Keep |
| installment_rate | ✅ Significant | Keep |
| existing_credits | ❌ Not significant | Drop on Day 2 |
| residence_since | ❌ Not significant | Drop on Day 2 |
| num_dependents | ❌ Not significant | Drop on Day 2 |

### Chi-Square (Categorical Features):
| Feature | Result | Action |
|---|---|---|
| checking_account | ✅ Significant (p≈0) | Keep — strongest feature |
| credit_history | ✅ Significant (p≈0) | Keep |
| savings_status | ✅ Significant (p≈0) | Keep |
| purpose | ✅ Significant | Keep |
| employment | ✅ Significant | Keep |
| personal_status | ✅ Significant | Keep |
| other_parties | ✅ Significant | Keep |
| property_magnitude | ✅ Significant | Keep |
| other_payment_plans | ✅ Significant | Keep |
| housing | ✅ Significant | Keep |
| foreign_worker | ✅ Significant | Keep* |
| job | ❌ Not significant | Regroup or drop Day 2 |
| own_telephone | ❌ Not significant | Drop Day 2 |

*foreign_worker: significant but fairness flag — handle carefully


# Day 1 — EDA Summary Report
**Dataset:** German Credit Data (UCI)  
**Date:** 2026-03-27  
**Objective:** Understand dataset structure, explore feature distributions, 
identify default patterns, validate findings statistically.

---

## 1. Dataset Overview
| Property | Value |
|---|---|
| Rows | 1,000 |
| Features | 20 input + 1 target |
| Numeric features | 7 (3 continuous, 4 ordinal) |
| Categorical features | 13 |
| Missing values | None |
| Class balance | 70% Good (700), 30% Bad (300) |
| Problem type | Binary classification |

---

## 2. Key EDA Findings

### Continuous Features
- **duration:** Right skewed, clusters at 12–24 months. 
  Bad borrowers take significantly longer loans (up to 72 months).
- **credit_amount:** Long right tail, 72 outliers (7.2%). 
  Amount alone is weak predictor — Amount ≠ Ability to Repay.
  Log transformation planned for Day 2.
- **age:** Bad borrowers peak at 24–25, Good at 26–27. 
  Younger borrowers default more.

### Categorical Features
- **checking_account:** 66.8% of borrowers have absent or 
  negative balance — most predictive feature.
- **credit_history:** "critical/other" surprisingly has lowest 
  default rate (17.1%) — experienced multi-credit borrowers 
  manage debt better.
- **savings_status:** Most borrowers have <100 DM savings — 
  very little financial buffer across the dataset.
- **foreign_worker:** 96.3% yes — near zero variance, 
  fairness flag.

### Multivariate Findings
- credit_amount × duration correlation: 0.62 — multicollinearity.
  Action: engineer monthly_burden = credit_amount/duration on Day 2.
- All numeric correlations with target < 0.25 — not linearly 
  separable. Tree-based models (XGBoost) preferred.
- Interaction: <0 checking + no credits = 76.92% default rate.
  Risk compounds, not just adds.

---

## 3. Default Borrower Profile
A borrower most likely to default shows these characteristics:
1. checking_status < 0 or 0<=X<200 — already in financial stress
2. credit_history: all paid / no credits — no active repayment track record
3. other_parties: co-applicant — shared responsibility dilutes accountability
4. savings_status: <100 or <500 DM — no financial buffer
5. property_magnitude: no known property — no collateral
6. duration >= 45 months — long loan = high uncertainty
7. age ~ 25–30 — early career, less financial stability

---

## 4. Hypothesis Testing Results
All findings validated with statistical tests (α = 0.05):

### Confirmed Significant Features (keep):
duration, credit_amount, age, installment_rate,
checking_account, credit_history, savings_status,
purpose, employment, personal_status, other_parties,
property_magnitude, other_payment_plans, housing,
foreign_worker*

### Confirmed Non-Significant Features (drop on Day 2):
residence_since, num_dependents, existing_credits,
own_telephone, job (candidate for regrouping)

*foreign_worker: significant but fairness flag

---

## 5. Day 2 Action Plan
| Action | Reason |
|---|---|
| Log transform credit_amount | Right skew, 7.2% outliers |
| Engineer credit_amount/duration | Multicollinearity 0.62, monthly burden proxy |
| Engineer checking × credit_history interaction | Compounds to 76.92% default rate |
| Engineer financial_stress_score | Combine checking + savings |
| Drop residence_since, num_dependents, existing_credits, own_telephone | Not statistically significant |
| Regroup job categories | 4 categories too broad, signal diluted |
| SMOTE after train-test split | 70/30 class imbalance |
| Fairness audit on personal_status, foreign_worker | Responsible AI — Unit V |